# Tutorial 03: Configurações Avançadas de Hardware (Reuse Factor & Profiling Numérico)

Neste tutorial, você aprenderá a ajustar os dois parâmetros de engenharia mais refinados do `hls4ml`:
1. **Fator de Reuso (Reuse Factor - $R$)**: O controle entre **área do chip** (número de multiplicadores/DSPs) e **latência** (ciclos de clock).
2. **Profiling Numérico de Camadas**: Inspecionar a distribuição estatística de ativações e pesos para configurar precisões heterogêneas por camada (evitando *overflow* e *underflow*).

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

sys.path.append('../..')
import models
import plotting
import hls4ml
from hls4ml.model.profiling import numerical

import torch

print(f"hls4ml: {hls4ml.__version__}")

## 1. Carregamento dos Dados e do Modelo Base
Carregamos as amostras de teste do LHC Jet Tagging e o modelo `JetTagger` treinado no Tutorial 01.

In [ ]:
data_dir = '../../data/jet-tagging'
X_test = np.load(os.path.join(data_dir, 'X_test.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test.npy'))
classes = np.load(os.path.join(data_dir, 'classes.npy'), allow_pickle=True)

eval_n = min(5000, len(X_test))

model = models.JetTagger()
weights_path = '../../models/pytorch_weights_part1.pt'
model.load_state_dict(torch.load(weights_path))
model.eval()

with torch.no_grad():
    y_torch = model(torch.tensor(X_test[:eval_n], dtype=torch.float32)).numpy()

acc_torch = accuracy_score(np.argmax(y_test[:eval_n], axis=1), np.argmax(y_torch, axis=1))
print(f"Acurácia PyTorch Baseline: {acc_torch * 100:.2f}%")

## 2. Parte 3A: Avaliando o Fator de Reuso (Reuse Factor = 1, 4, 8)
O parâmetro `ReuseFactor` define quantas vezes um multiplicador físico de silício é reutilizado sequencialmente:
- **`ReuseFactor = 1`**: Máximo paralelismo (todas as contas no mesmo ciclo). Alta área, latência mínima.
- **`ReuseFactor = 4`**: Os multiplicadores são reaproveitados em 4 ciclos de clock. Economiza ~75% dos DSPs.
- **`ReuseFactor = 8`**: Economiza ~87.5% dos DSPs.

Abaixo geramos e simulamos as 3 versões com `g++` para demonstrar que a matemática e a acurácia permanecem idênticas.

In [ ]:
rf_results = {}

for rf in [1, 4, 8]:
    cfg = hls4ml.utils.config_from_pytorch_model(
        model,
        input_shape=(16,),
        granularity='model',
        backend='Vivado',
    )
    cfg['Model']['Precision']['default'] = 'ap_fixed<16,6>'
    cfg['Model']['ReuseFactor'] = rf
    
    out_dir = f"hls_prj_reuse_{rf}"
    h_model = hls4ml.converters.convert_from_pytorch_model(
        model,
        hls_config=cfg,
        output_dir=out_dir,
        part='xcu200-fsgd2104-2-e'
    )
    h_model.compile()
    y_rf = h_model.predict(X_test[:eval_n])
    acc_rf = accuracy_score(np.argmax(y_test[:eval_n], axis=1), np.argmax(y_rf, axis=1))
    rf_results[rf] = acc_rf
    print(f"ReuseFactor = {rf:<2} -> Acurácia: {acc_rf * 100:.2f}%")

## 3. Parte 3B: Profiling Numérico de Pesos e Ativações
Ativamos `Trace = True` nas camadas e usamos `numerical` para gerar os gráficos de faixas numéricas de cada camada.

In [ ]:
prof_cfg = hls4ml.utils.config_from_pytorch_model(
    model,
    input_shape=(16,),
    granularity='name',
    backend='Vivado',
)
for layer in prof_cfg['LayerName'].keys():
    prof_cfg['LayerName'][layer]['Trace'] = True

hls_prof = hls4ml.converters.convert_from_pytorch_model(
    model,
    hls_config=prof_cfg,
    output_dir='hls_prj_profiling',
    part='xcu200-fsgd2104-2-e'
)
hls_prof.compile()

# Executar profiling visual
plt.figure(figsize=(12, 6))
numerical(model=model, hls_model=hls_prof, X=X_test[:500])
plt.show()

## 4. Parte 3C: Configuração de Precisão Heterogênea por Camada
Com base no profiling, podemos definir larguras de bits diferentes para cada camada:
- `fc1`: Usar `ap_fixed<8,2>` (camada inicial tolerante a ruído numérico).
- `output`: Usar `ap_fixed<16,6>` (camada final que decide as probabilidades de classe).

In [ ]:
custom_cfg = hls4ml.utils.config_from_pytorch_model(
    model,
    input_shape=(16,),
    granularity='name',
    backend='Vivado',
)
custom_cfg['LayerName']['fc1']['Precision']['weight'] = 'ap_fixed<8,2>'
custom_cfg['LayerName']['fc1']['Precision']['result'] = 'ap_fixed<8,2>'
custom_cfg['LayerName']['output']['Precision']['weight'] = 'ap_fixed<16,6>'
custom_cfg['LayerName']['output']['Precision']['result'] = 'ap_fixed<16,6>'

hls_custom = hls4ml.converters.convert_from_pytorch_model(
    model,
    hls_config=custom_cfg,
    output_dir='hls_prj_custom_precision',
    part='xcu200-fsgd2104-2-e'
)
hls_custom.compile()

y_custom = hls_custom.predict(X_test[:eval_n])
acc_custom = accuracy_score(np.argmax(y_test[:eval_n], axis=1), np.argmax(y_custom, axis=1))

print(f"Acurácia Homogênea 16-bit : {rf_results[1] * 100:.2f}%")
print(f"Acurácia Heterogênea (8b/16b) : {acc_custom * 100:.2f}%")
print(f"Diferença de precisão: {abs(rf_results[1] - acc_custom) * 100:.2f}%")